In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np
import math
import tensorflow_datasets as tfds
from keras.src.layers import RNN
from sketchrnn.SketchRNN import SketchRNN
from dataset.visualization.util import rasterize, visualize_sample, getSample
from dataset.preparation.util import  strokes_to_sketchrnn, sketchrnn_to_strokes, normalize_strokes, unnormalize_strokes, pad_sketchrnn, check_stats

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

%load_ext autoreload
%autoreload 2

In [ ]:
import tensorflow as tf
print("TF:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", tf.config.list_physical_devices("GPU"))


# Visualizing Data

In [ ]:
img_list, _ = getSample(16)
visualize_sample(img_list)

# Preparing dataset

In [ ]:
img_list, stroke_list = getSample(16)
sketchrnn_seqs = [strokes_to_sketchrnn(stroke) for stroke in stroke_list]
sketchrnn_seqs[0]


In [ ]:
original_stroke = sketchrnn_to_strokes(sketchrnn_seqs[0])

In [ ]:
rasterize(original_stroke, size=256, stroke_width=5)

In [ ]:
rasterize(stroke_list[0], size=256, stroke_width=5)

# Data Analysis

In [ ]:
img_list, stroke_list = getSample(200)
sketchrnn_seqs = [strokes_to_sketchrnn(stroke) for stroke in stroke_list]
padded_seqs, mask = pad_sketchrnn(sketchrnn_seqs)

padded_seqs.shape, mask.shape

In [ ]:
original_stroke = sketchrnn_to_strokes(padded_seqs[0])
rasterize(original_stroke, size=256, stroke_width=5)

In [ ]:
check_stats(padded_seqs, mask)


In [ ]:
from dataset.preparation.util import compute_stats_single, normalize_strokes_single


ns, mean, std = normalize_strokes_single(padded_seqs, mask)
compute_stats_single(ns, mask)
# mean_x, mean_y, std_x, std_y
# np.max(ns[:, :,0]), np.min(ns[:, :,0]), np.max(ns[:, :,1]), np.min(ns[:, :,1])

# data Cleaning

In [ ]:
# from sklearn.decomposition import PCA
# from sklearn.cluster import KMeans
# size = 50000
# img_list, stroke_list = getSample(size)



# stroke_list[0][0]




In [ ]:
# imgs = np.stack([np.array(d) for d in img_list], axis=0)  
# X = imgs.reshape(len(imgs), -1).astype(np.float32) / 255.0 
# pca = PCA(n_components=64, random_state=0)
# Z = pca.fit_transform(X)                                                 # (N,64)

# # Cluster
# k = 10
# km = KMeans(n_clusters=k, random_state=0, n_init="auto")
# labels = km.fit_predict(Z)

In [ ]:
# targets = [6,0]
# counts = {k: np.sum(labels == k) for k in targets}
# print(counts)

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# def show_cluster(imgs, labels, cluster_id, n=25):
#     idx = np.where(labels == cluster_id)[0]
    
#     if len(idx) == 0:
#         print(f"Cluster {cluster_id} is empty")
#         return
    
#     idx = np.random.choice(idx, min(n, len(idx)), replace=False)
    
#     cols = int(np.sqrt(n))
#     rows = int(np.ceil(n / cols))
    
#     plt.figure(figsize=(cols, rows))
    
#     for i, j in enumerate(idx):
#         plt.subplot(rows, cols, i+1)
#         plt.imshow(imgs[j], cmap='gray')
#         plt.axis('off')
    
#     plt.suptitle(f"Cluster {cluster_id} ({len(idx)} samples)")
#     plt.show()

# num_clusters = len(np.unique(labels))

# for k in range(num_clusters):
#     show_cluster(imgs, labels, k, n=16)
#     pass


In [ ]:
# indices = np.where(np.isin(labels, targets))[0]
# filtered_drawings = [stroke_list[i] for i in indices]

# Building Model

# Training

In [ ]:
size = 120000
img_list, stroke_list = getSample(size)
seqs = [strokes_to_sketchrnn(stroke) for stroke in stroke_list]
# data, masks = pad_sketchrnn(seqs)
# # data, mean_x, mean_y, std_x, std_y = normalize_strokes(data, masks)
# masks = masks.astype(np.float32)
# data.shape, masks.shape

# mean_x, mean_y, std_x, std_y
seq_length = [seq.shape[0] for seq in seqs]
print(min(seq_length), max(seq_length))

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "index": range(len(seq_length)),
    "length": seq_length
})

df.hist(column='length', bins=50)


In [ ]:
p25, p75 = df['length'].quantile([0.25, 0.75]).values.tolist()
IQR = p75 - p25
lower_bound = p25 - 1.5 * IQR
upper_bound = p75 + 1.5 * IQR
df = df[(df['length'] >= lower_bound) & (df['length'] <= upper_bound)]
df.boxplot(column='length')

In [ ]:

filted_index = df['index'].values.tolist()

In [ ]:
filtered_drawings = [strokes_to_sketchrnn(stroke_list[i])   for i in filted_index]
data, masks = pad_sketchrnn(filtered_drawings)
data, mean, std = normalize_strokes_single(data, masks)


data.shape, masks.shape

In [ ]:
# import random

# # unnormalize_data = unnormalize_strokes(data, mean_x, mean_y, std_x, std_y)
# img_list = []
# drawing = unnormalize_strokes(data, mean_x, mean_y, std_x, std_y)
# for d in drawing:
#     drawing = sketchrnn_to_strokes(d)   # where result[b] is (T,5)
#     img = rasterize(drawing, size=256, stroke_width=5)
#     img_list.append(img)
# i = random.randrange(8000)
# visualize_sample(img_list[i:i+16])

In [ ]:
compute_stats_single(data, masks)

In [ ]:
def create_partial_sketch(data, mask):
    encoder_mask = np.copy(mask)
    decoder_mask = np.copy(mask)
    decoder_data = np.copy(data)
    partial_data = np.copy(data)
    for i in range(data.shape[0]):
        seq_len = int(encoder_mask[i].sum())
        if seq_len > 1:
            cut_point = np.random.randint(seq_len//2, seq_len)  # Ensure at least one point remains
            partial_data[i, cut_point:, :] = 0.0
            encoder_mask[i, cut_point:] = 0.0  # Update mask to reflect the cut
    return partial_data, encoder_mask, decoder_data, decoder_mask

In [ ]:
N = len(data)
split = int(0.8 * N)

# train_data = data[:split]
# val_data   = data[split:]
train_encoder_data, train_encoder_mask, train_decoder_data, train_decoder_mask = create_partial_sketch(data[:split], masks[:split])
val_encoder_data, val_encoder_mask, val_decoder_data, val_decoder_mask = create_partial_sketch(data[split:], masks[split:])
y_train = train_decoder_data

X_train = {
    "encoder_data": train_encoder_data,
    "encoder_mask": train_encoder_mask,
    "decoder_data": train_decoder_data,
    "decoder_mask": train_decoder_mask,
    
}

y_val = val_decoder_data

# val_data   = data[split:]
# y_val   = data[split:]
noise = np.random.uniform(low=0.9, high=1.1, size=(val_decoder_data.shape[0], 1, 2)).astype(np.float32)
val_decoder_data[:, :, :2] = val_decoder_data[:, :, :2] * noise
val_encoder_data[:, :, :2] = val_encoder_data[:, :, :2] * noise
# y_val[:,:,:2] = y_val[:, :, :2] * np.random.uniform(low=0.9, high=1.1, size=(y_val.shape[0], 1, 2)).astype(np.float32)
val_mask   = masks[split:]
X_val = {
    "encoder_data": val_encoder_data,
    "encoder_mask": val_encoder_mask,
    "decoder_data": val_decoder_data,
    "decoder_mask": val_decoder_mask
}

display(X_train["encoder_data"].shape, X_train["encoder_mask"].shape, X_train["decoder_data"].shape, X_train["decoder_mask"].shape)
# display(y_train.shape)
display(X_val["encoder_data"].shape, X_val["encoder_mask"].shape, X_val["decoder_data"].shape, X_val["decoder_mask"].shape)
# display(y_val.shape)

In [ ]:
train_encoder_data.shape, train_decoder_data.shape

In [ ]:
is_zero = tf.reduce_all(val_encoder_data == 0, axis=-1)

has_zero = tf.reduce_any(is_zero, axis=1)

first_zero_idx = tf.argmax(tf.cast(is_zero, tf.int32), axis=1)

first_zero_idx = tf.where(has_zero, first_zero_idx, -1)
first_zero_idx

In [ ]:
is_zero = tf.logical_not(val_encoder_mask)  # or train_encoder_mask == False

has_zero = tf.reduce_any(is_zero, axis=1)

first_zero_idx = tf.argmax(tf.cast(is_zero, tf.int32), axis=1)

first_zero_idx = tf.where(has_zero, first_zero_idx, -1)
first_zero_idx

In [ ]:
is_zero = tf.reduce_all(val_decoder_data == 0, axis=-1)

has_zero = tf.reduce_any(is_zero, axis=1)

first_zero_idx = tf.argmax(tf.cast(is_zero, tf.int32), axis=1)

first_zero_idx = tf.where(has_zero, first_zero_idx, -1)
first_zero_idx

In [ ]:
is_zero = tf.logical_not(val_decoder_mask)  # or train_encoder_mask == False

has_zero = tf.reduce_any(is_zero, axis=1)

first_zero_idx = tf.argmax(tf.cast(is_zero, tf.int32), axis=1)

first_zero_idx = tf.where(has_zero, first_zero_idx, -1)
first_zero_idx

In [ ]:
GLOBAL_BATCH_SIZE = 100*4
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)) \
    .batch(GLOBAL_BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)
val_ds =  (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .shuffle(buffer_size=len(X_val), reshuffle_each_iteration=True)
    .batch(GLOBAL_BATCH_SIZE)
    .take(15)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
M=10
z_dim=128
enc_hidden=512
dec_hidden=1024
enc_cell_type='lstm'
dec_cell_type='lstm'
R=0.99999
wKL=1
KLmin=0.2

# 20 128 256 128


In [ ]:
from datetime import datetime
import time

from sketchrnn.decoder import Decoder
from sketchrnn.encoder import Encoder
id = datetime.now().strftime("%Y%m%d-%H%M%S")
best_ckpt = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"checkpoints/{str(id)}/best.keras",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    verbose=1,
    
)

best_recon = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"checkpoints/{str(id)}/best_recon.keras",
    monitor="val_reconstruction_loss",
    save_best_only=True,
    save_weights_only=False,
    verbose=1,
    mode="min"
)

best_kl = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"checkpoints/{str(id)}/best_kl.keras",
    monitor="val_kl_loss",
    save_best_only=True,
    save_weights_only=False,
    verbose=1,
    mode="min",
    
)

latest_ckpt = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"checkpoints/{str(id)}/latest.keras",
    save_best_only=False,
    save_weights_only=False,
    verbose=1
)

config = {
    "M": M,
    "z_dim": z_dim,
    "enc_hidden": enc_hidden,
    "dec_hidden": dec_hidden,
    "enc_cell_type": enc_cell_type,
    "dec_cell_type": dec_cell_type,
    'R': R,
    'wKL': wKL,
    'KLmin': KLmin
}

batch_size = 100
initial_lr = 1e-3
final_lr = 1e-5
epochs = 300
steps_per_epoch = len(train_encoder_data) // batch_size

decay_steps = epochs * steps_per_epoch
decay_rate = final_lr / initial_lr

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=initial_lr,
    decay_steps=decay_steps,
    decay_rate=decay_rate,
    staircase=False
)

# lr_scheduler = tf.keras.optimizers.schedules.cosine_decay(
#     initial_learning_rate=initial_lr,
#     decay_steps=decay_steps,
#      alpha=final_lr / initial_lr,   

log_dir = f"checkpoints/{id}/logs/" 
log_cb = tf.keras.callbacks.TensorBoard(log_dir=log_dir,  update_freq=50)

import json


# strategy = tf.distribute.MirroredStrategy()
# with strategy.scope():
model = SketchRNN(M=M, z_dim=z_dim, enc_hidden=enc_hidden, dec_hidden=dec_hidden, enc_cell_type=enc_cell_type, dec_cell_type=dec_cell_type, R=R, wKL=wKL, KLmin=KLmin)

dummy_data = tf.zeros([1, 2, 5])
dummy_mask = tf.ones([1, 2], dtype=tf.bool)
inputs = {
    "data": dummy_data,
    "mask": dummy_mask,

}

model(inputs, training=False)


optimizer = optimizer = tf.keras.optimizers.AdamW(
learning_rate=lr_schedule,
weight_decay=0.01,
clipnorm=1.0
)
model.compile(optimizer=optimizer)
# model.load_model(f"./checkpoints/{str('20260221-050950')}/latest.keras")
os.makedirs(f"checkpoints/{str(id)}", exist_ok=True)
with open(f"checkpoints/{str(id)}/config.json", "w") as f:
    json.dump(config, f, indent=2)
# print(model.R)
history = model.fit(train_ds, epochs=epochs, verbose=1, callbacks=[ log_cb, latest_ckpt, best_ckpt, best_recon, best_kl], validation_data=val_ds)

In [ ]:
def plot_loss(history, detail=False):
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    if detail:
        plt.plot(history.history['kl_loss'], label='kl_loss')
        plt.plot(history.history['reconstruction_loss'], label='reconstruction_loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()
plot_loss(history, True)

#seem like we are overfitting when the KL floored

In [ ]:
model.build(input_shape={"data": (None, None, 5), "mask": (None, None)})
model.save(f"checkpoints/{id}/latest.keras")

# Test reconstruction

In [ ]:
from dataset.preparation.util import unnormalize_strokes_single
from eval.util import fit_quickdraw_to_canvas

# id = '20260222-082332'
# id = '20260302-135821'
# id ='20260302-014909'
id = '20260302-014909'
model = SketchRNN(M=M, z_dim=z_dim, enc_hidden=enc_hidden, dec_hidden=dec_hidden, enc_cell_type=enc_cell_type, dec_cell_type=dec_cell_type)
dummy_data = tf.zeros([1, 2, 5])
dummy_mask = tf.ones([1, 2], dtype=tf.bool)
inputs = {
        "data": dummy_data,
        "mask": dummy_mask
    }   
model(inputs, training=False)
# print('20260112-075039')
model.load_weights(f"checkpoints/{str(id)}/best_recon.keras")
idx = np.random.choice(val_encoder_data.shape[0], size=16, replace=False)
# idx = np.array([906, 124, 734, 139, 973, 346, 125, 984, 424, 801, 106, 288, 925,
#        416, 917, 214])
val_enc = val_decoder_data[idx, :, :]
val_enc_mask = val_decoder_mask[idx, :]
val_dec = val_decoder_data[idx, :, :]
val_dec_mask = val_decoder_mask[idx, :]


result = model.recon(({"encoder_data": val_enc, "decoder_data": val_dec, "encoder_mask": val_enc_mask, "decoder_mask": val_dec_mask}, None), training=False)
result = np.concatenate([val_dec[:, :1, :], result], axis=1)  # prepend the initial point
result = unnormalize_strokes_single(result, mean, std)

pen_end = result[:, :, 4]

eos_mask = pen_end > 0.5
T = tf.shape(result)[1]                      # int32
first_eos = tf.argmax(eos_mask, axis=1, output_type=tf.int32)  # int32
has_eos = tf.reduce_any(eos_mask, axis=1)     # bool

lengths = tf.where(
    has_eos,
    first_eos + 1,
    tf.fill(tf.shape(first_eos), T)           # int32 vector of T
)

ragged = tf.RaggedTensor.from_tensor(result, lengths=lengths)
drawings = ragged.to_list()
# drawings = result.tolist()
img_list = []
# print(mean_x, mean_y, std_x, std_y)
unnormalize_data = unnormalize_strokes_single(val_dec, mean, std)
for d in unnormalize_data:
    drawing = sketchrnn_to_strokes(d)   # where result[b] is (T,5)
    img = rasterize(drawing, size=256, stroke_width=5)
    img_list.append(img)
visualize_sample(img_list)

img_list = []
abs_drawings = []
for d in drawings:
    drawing = sketchrnn_to_strokes(d)   # where result[b] is (T,5)
    drawing = fit_quickdraw_to_canvas(drawing)
    abs_drawings.append(drawing)
    img = rasterize(drawing, size=256, stroke_width=5)
    img_list.append(img)
visualize_sample(img_list)


In [ ]:
abs_drawings[0]

# Test Generation

In [ ]:
from eval.util import fit_quickdraw_to_canvas

id ='20260302-014909' # good generation
model = SketchRNN(M=M, z_dim=z_dim, enc_hidden=enc_hidden, dec_hidden=dec_hidden, enc_cell_type=enc_cell_type, dec_cell_type=dec_cell_type)
dummy_data = tf.zeros([1, 2, 5])
dummy_mask = tf.ones([1, 2], dtype=tf.bool)
inputs = {
        "data": dummy_data,
        "mask": dummy_mask
    }   
model(inputs, training=False)
# print('20260112-075039')
model.load_weights(f"checkpoints/{str(id)}/latest.keras")
idx = np.random.choice(val_encoder_data.shape[0], size=1, replace=False)
# idx = np.array([906, 124, 734, 139, 973, 346, 125, 984, 424, 801, 106, 288, 925,
#        416, 917, 214])
idx = np.array([502])
val_enc = val_encoder_data[idx, :, :]
val_enc_mask = val_encoder_mask[idx, :]
val_dec = val_decoder_data[idx, :, :]
val_dec_mask = val_decoder_mask[idx, :]

gen_list = []

img_list = []
unnormalize_data = unnormalize_strokes_single(val_encoder_data[idx, :, :], mean, std)
for d in unnormalize_data:
    drawing = sketchrnn_to_strokes(d)   # where result[b] is (T,5)
    img = rasterize(drawing, size=256, stroke_width=5)
    img_list.append(img)
visualize_sample(img_list)
    
for i in range(16):
    result = model.generate(({"encoder_data": val_enc, "decoder_data": val_dec, "encoder_mask": val_enc_mask, "decoder_mask": val_dec_mask}, None), pen_temp=0.7, gmm_temp=1, disabled_pen_end=False, training=False)
    result = np.concatenate([val_dec[:, :1, :], result], axis=1)  # prepend the initial point

    # print(data.shape, result.shape)
    
    # visualize_sample(img_list)
    print(result.shape)
    re_gen = np.concatenate([val_encoder_data[idx, -1:, :], result], axis=1)  # prepend the initial point
    # re_gen = unnormalize_strokes_single(re_gen, mean, std)
    img_list = []
    abs_drawings = []
    for d in re_gen:
        drawing = sketchrnn_to_strokes(d)   # where result[b] is (T,5)
        drawing = fit_quickdraw_to_canvas(drawing)
        abs_drawings.append(drawing)
        img = rasterize(drawing, size=256, stroke_width=5, gen=True, gen_bound=val_encoder_data.shape[1])
        gen_list.append(img)
visualize_sample(gen_list)

# data.shape, val_data.shape

In [ ]:
data[0], val_data[idx[0], -10:, :]

In [ ]:
re_gen[1][:40]

In [ ]:
result[0][20:25]

In [ ]:
re_gen[0][:21]